## **Expiry-Day Liquidity Concentration in Indian Index Derivatives**

### **Notebook: 01: Data Collection**

---

**Scope:** Downloads Bank Nifty and Nifty 50 index OHLCV, constructs the settlement-weekday expiry calendar and trading holiday list, and (one-time, documented but not re-run here) pulls the full NSE futures-and-options Bhavcopy archive.

---

### Pipeline sequence

1. **01_data_collection.ipynb** -- index OHLCV, expiry calendar, holidays, F&O Bhavcopy archive
2. **02_feature_engineering_and_fo_construction.ipynb** -- technical/volatility features; lot-size-invariant turnover and open interest
3. **03_weekly_options_discontinuation_identification.ipynb** -- dating the Nov 2024 weekly-contract removal directly from exchange records
4. **04_primary_diff_in_diff_analysis.ipynb** -- the headline difference-in-differences result (Bank Nifty vs. Nifty 50)
5. **05_robustness_checks.ipynb** -- placebo cutover dates, leave-one-out, event-study leads/lags
6. **06_expiry_volume_event_study.ipynb** -- descriptive expiry-day volume/OI patterns by regime
7. **07_walk_forward_predictability.ipynb** -- secondary analysis: leakage-free return-predictability test (null result)
8. **08_interrupted_time_series.ipynb** -- secondary analysis: settlement-weekday effect on returns/volatility (null result)
9. **09_figures_for_paper.ipynb** -- regenerates all manuscript figures

**Current location:** **01_data_collection.ipynb** <- *you are here*


### 1. Index OHLCV, expiry calendar, and holidays

`src.data_loader.load_data_pipeline` downloads daily Bank Nifty OHLCV from Yahoo Finance, programmatically builds the monthly expiry calendar across the Thursday -> Wednesday -> Tuesday settlement-weekday reforms (rolling back for exchange holidays), and derives the holiday list from the trading calendar itself.

In [1]:
import sys
from pathlib import Path

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

import pandas as pd
import json
from src import config

pd.set_option('display.max_columns', 20)
print('Project root:', project_dir)

Project root: E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml


In [2]:
from src.data_loader import load_data_pipeline

df_ohlcv, df_expiry, df_holidays = load_data_pipeline()
print(f'OHLCV: {len(df_ohlcv)} rows, {df_ohlcv["Date"].min()} to {df_ohlcv["Date"].max()}')
print(f'Expiry calendar: {len(df_expiry)} months')
print(f'Holidays: {len(df_holidays)} days')
df_expiry.tail(10)

[*********************100%***********************]  1 of 1 completed

Saved raw Bank Nifty data to E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml\data\raw\banknifty_ohlcv_2023_2026.csv
Generated and saved expiry calendar to E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml\data\raw\nse_expiry_calendar.csv
Identified and saved 93 holidays to E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml\data\raw\nse_holidays.csv
OHLCV: 1680 rows, 2020-01-01 to 2026-10-16
Expiry calendar: 82 months
Holidays: 93 days


,Year,Month,ScheduledExpiry,ActualExpiry,Rule,HolidayAdjusted
72,2026,1,2026-01-27,2026-01-27,Last Tuesday,0
73,2026,2,2026-02-24,2026-02-24,Last Tuesday,0
74,2026,3,2026-03-31,2026-03-30,Last Tuesday,1
75,2026,4,2026-04-28,2026-04-28,Last Tuesday,0
76,2026,5,2026-05-26,2026-05-26,Last Tuesday,0
77,2026,6,2026-06-30,2026-06-30,Last Tuesday,0
78,2026,7,2026-07-28,2026-07-28,Last Tuesday,0
79,2026,8,2026-08-25,2026-08-25,Last Tuesday,0
80,2026,9,2026-09-29,2026-09-29,Last Tuesday,0
81,2026,10,2026-10-27,2026-10-27,Last Tuesday,0


### 2. NSE futures-and-options Bhavcopy archive

The primary analysis needs daily per-contract volume, traded value, and open interest for every Bank Nifty and Nifty 50 futures/options contract, which the index OHLCV above does not provide. This comes from NSE's own daily Bhavcopy archive, spanning a legacy CSV format (pre 2024-07-08) and the newer UDiFF format.

Downloading the full 2020-2026 archive (~1,700 trading days) is a one-time, rate-limited step and is **not re-run in this notebook**. To (re)build it from scratch:

```bash
python -m scripts.download_fo_bhavcopy 2020-01-01 2026-10-18
```

The cell below just reports what is already on disk.

In [3]:
legacy_dir = config.RAW_DATA_DIR / 'fo_bhavcopy' / 'legacy'
udiff_dir = config.RAW_DATA_DIR / 'fo_bhavcopy' / 'udiff'
n_legacy = len(list(legacy_dir.glob('*.zip'))) if legacy_dir.exists() else 0
n_udiff = len(list(udiff_dir.glob('*.zip'))) if udiff_dir.exists() else 0
print(f'Legacy-format Bhavcopy files on disk: {n_legacy}')
print(f'UDiFF-format Bhavcopy files on disk: {n_udiff}')
print(f'Total: {n_legacy + n_udiff}')

Legacy-format Bhavcopy files on disk: 1115
UDiFF-format Bhavcopy files on disk: 564
Total: 1679
